# Exercise 02: Schema Design & Constraints

One of the most important things you can do to protect data quality in a warehouse is to encode your **business rules directly into the schema**.

In this exercise you'll design a schema for a company's HR system, applying six different types of constraints to enforce data quality at the database level.

---

### The six constraints you'll use

| Constraint | What it does | Example |
|---|---|---|
| `PRIMARY KEY` | Uniquely identifies each row. Cannot be NULL or duplicate. | `employee_id INTEGER PRIMARY KEY` |
| `FOREIGN KEY` | Links a column to a primary key in another table. Prevents orphaned rows. | `REFERENCES departments(department_id)` |
| `UNIQUE` | No two rows can have the same value. Can be NULL. | `email VARCHAR UNIQUE` |
| `NOT NULL` | The column must always have a value. | `name VARCHAR NOT NULL` |
| `CHECK` | Value must pass a condition. | `salary CHECK (salary >= 0)` |
| `DEFAULT` | Value used when none is provided on INSERT. | `status VARCHAR DEFAULT 'active'` |

---

### The schema you'll build

```
Schema: company
  departments   — one row per department
  employees     — one row per employee, linked to a department
```

## Setup

In [ ]:
import duckdb

con = duckdb.connect()
print('Connected! DuckDB version:', duckdb.__version__)

---
## Task 1: Create a Schema

A **schema** is a namespace that groups related tables together.
In production warehouses you'll commonly see schemas like `raw`, `staging`, `marts`, `finance`, `hr`, etc.

Create a schema called `company`.

```sql
CREATE SCHEMA schema_name;
```

To create a table inside a schema, prefix the table name: `company.departments`.

In [ ]:
# ✏️ Create the 'company' schema
con.execute('''

''')

In [ ]:
# Check your work — 'company' should appear in the list
con.execute('''
    SELECT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
''').df()

---
## Task 2: Create the `departments` table

Design the `company.departments` table with the following columns and constraints:

| Column | Type | Constraints | Notes |
|---|---|---|---|
| `department_id` | `INTEGER` | `PRIMARY KEY` | Unique identifier |
| `name` | `VARCHAR` | `NOT NULL`, `UNIQUE` | No two departments can share a name |
| `budget` | `DECIMAL(15, 2)` | `NOT NULL`, `CHECK (budget > 0)` | Must be a positive number |

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE company.departments (

        -- ✏️ Define your columns and constraints here

    )
''')

In [ ]:
# Check your table structure
con.execute('''
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'company' AND table_name = 'departments'
    ORDER BY ordinal_position
''').df()

---
## Task 3: Create the `employees` table

Design the `company.employees` table. This table uses all six constraint types.

| Column | Type | Constraints | Notes |
|---|---|---|---|
| `employee_id` | `INTEGER` | `PRIMARY KEY` | Unique identifier |
| `first_name` | `VARCHAR` | `NOT NULL` | |
| `last_name` | `VARCHAR` | `NOT NULL` | |
| `email` | `VARCHAR` | `NOT NULL`, `UNIQUE` | No duplicate email addresses |
| `department_id` | `INTEGER` | `REFERENCES company.departments(department_id)` | Foreign key |
| `salary` | `DECIMAL(10, 2)` | `NOT NULL`, `CHECK (salary >= 0)` | Must be non-negative |
| `hire_date` | `DATE` | `NOT NULL`, `DEFAULT CURRENT_DATE` | Defaults to today |
| `employment_status` | `VARCHAR` | `NOT NULL`, `DEFAULT 'active'`, `CHECK (employment_status IN ('active', 'inactive', 'on-leave'))` | Only three allowed values |

In [ ]:
con.execute('''
    CREATE OR REPLACE TABLE company.employees (

        -- ✏️ Define your columns and constraints here

    )
''')

In [ ]:
# Check your table structure
con.execute('''
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'company' AND table_name = 'employees'
    ORDER BY ordinal_position
''').df()

---
## Task 4: Insert Data

Populate both tables with some sample data.
Departments must be inserted before employees (because of the foreign key).

In [ ]:
# Insert departments first
con.execute('''
    INSERT INTO company.departments (department_id, name, budget) VALUES
        (1, 'Engineering',  2500000.00),
        (2, 'Marketing',     800000.00),
        (3, 'Operations',   1200000.00),
        (4, 'Data',          950000.00)
''')

con.execute('SELECT * FROM company.departments').df()

In [ ]:
# ✏️ Insert at least 5 employees into company.employees
# Notice: you don't need to supply hire_date or employment_status — the DEFAULTs will fill them in
con.execute('''
    INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
    VALUES
        -- ✏️ Add your rows here
        -- Example: (1, 'Alice', 'Smith', 'alice@company.com', 1, 95000.00)

''')

con.execute('SELECT * FROM company.employees').df()

---
## Task 5: Test Your Constraints

Each cell below tries to INSERT data that should be rejected.
Run each one and observe the error message. This is how constraints protect your data!

> These cells are **expected to fail**. Read the error message — it tells you exactly which constraint was violated.

In [ ]:
# Test PRIMARY KEY — inserting a duplicate employee_id should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (1, 'Duplicate', 'Person', 'dup@company.com', 1, 50000)
    ''')
    print('❌ Insert succeeded — PRIMARY KEY not enforced!')
except Exception as e:
    print('✅ PRIMARY KEY violation caught:', e)

In [ ]:
# Test CHECK constraint — a negative salary should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (999, 'Bad', 'Salary', 'bad@company.com', 1, -100)
    ''')
    print('❌ Insert succeeded — CHECK constraint not enforced!')
except Exception as e:
    print('✅ CHECK violation caught:', e)

In [ ]:
# Test FOREIGN KEY — referencing a department_id that doesn't exist should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (998, 'Lost', 'Employee', 'lost@company.com', 999, 60000)
    ''')
    print('❌ Insert succeeded — FOREIGN KEY not enforced!')
except Exception as e:
    print('✅ FOREIGN KEY violation caught:', e)

In [ ]:
# Test UNIQUE — duplicate email address should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (997, 'Same', 'Email', 'alice@company.com', 1, 60000)
    ''')
    print('❌ Insert succeeded — UNIQUE constraint not enforced!')
except Exception as e:
    print('✅ UNIQUE violation caught:', e)

In [ ]:
# Test CHECK on employment_status — an invalid status value should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary, employment_status)
        VALUES (996, 'Bad', 'Status', 'status@company.com', 1, 60000, 'fired')
    ''')
    print('❌ Insert succeeded — CHECK constraint on employment_status not enforced!')
except Exception as e:
    print('✅ CHECK violation caught:', e)

---
## Task 6: Explore Constraints in the Information Schema

Just as you can query `information_schema.columns` to see column definitions, you can query DuckDB's built-in functions to inspect constraints.

In [ ]:
# View all constraints across our schema
con.execute('''
    SELECT
        schema_name,
        table_name,
        constraint_index,
        constraint_type,
        constraint_text
    FROM duckdb_constraints()
    WHERE schema_name = 'company'
    ORDER BY table_name, constraint_index
''').df()

In [ ]:
# View the final state of your employees table
con.execute('''
    SELECT
        e.employee_id,
        e.first_name,
        e.last_name,
        e.email,
        d.name AS department,
        e.salary,
        e.hire_date,
        e.employment_status
    FROM company.employees e
    JOIN company.departments d ON e.department_id = d.department_id
    ORDER BY e.employee_id
''').df()

---
## Summary

| Constraint | Why it matters |
|---|---|
| `PRIMARY KEY` | Every row is uniquely addressable — essential for JOINs |
| `FOREIGN KEY` | Prevents orphaned rows — referential integrity |
| `UNIQUE` | Prevents duplicates on business keys (emails, codes) |
| `NOT NULL` | Ensures critical fields always have values |
| `CHECK` | Encodes business rules directly in the schema |
| `DEFAULT` | Makes inserts simpler and reduces NULL surprises |

These constraints are your **first line of defence** against bad data entering your warehouse.